# Real Simple Benchmark: GAP + MILP + Genetic

Ноутбук запускает 3 алгоритма на уменьшенном реальном датасете `real_simple`.
Если `real_simple` отсутствует, он будет собран автоматически из `dataset_real_spb_ready.json`.


In [21]:
from __future__ import annotations

from pathlib import Path
import sys
import importlib
import inspect
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Принудительно вычищаем старые flowopt-модули из kernel-кеша
for mod_name in list(sys.modules):
    if mod_name.startswith("flowopt"):
        sys.modules.pop(mod_name, None)

import flowopt.pipeline as fp
from flowopt.backend import RealSimpleBuildConfig, build_real_simple_dataset

fp = importlib.reload(fp)

DATA_ROOT = fp.DATA_ROOT
run_real_gap_vrp = fp.run_real_gap_vrp
run_real_milp = fp.run_real_milp
run_real_genetic = fp.run_real_genetic

# Явная проверка, что подхватился новый real_milp API
milp_sig = inspect.signature(run_real_milp)
if "time_limit_sec" not in milp_sig.parameters:
    raise RuntimeError(f"Stale run_real_milp loaded: signature={milp_sig}")

pd.set_option("display.max_colwidth", 160)
print("REPO_ROOT:", REPO_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("pipeline module:", fp.__file__)
print("run_real_milp signature:", milp_sig)


REPO_ROOT: /Users/igoreshka/Desktop/Optimization-of-flows
DATA_ROOT: /Users/igoreshka/Desktop/Optimization-of-flows/src/data
pipeline module: /Users/igoreshka/Desktop/Optimization-of-flows/src/flowopt/pipeline.py
run_real_milp signature: (*, dataset_path: 'Path | str' = PosixPath('/Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/real_simple/dataset_real_spb_simple.json'), time_limit_sec: 'int' = 60, unassigned_penalty: 'float' = 100000.0, show_progress: 'bool' = False, progress_hook: 'Callable[[str], None] | None' = None) -> 'RunMetrics'


In [22]:
import json

REAL_DIR = DATA_ROOT / "real"
BASE_DATASET_PATH = REAL_DIR / "dataset_real_spb_ready.json"
SIMPLE_DIR = REAL_DIR / "real_simple"
SIMPLE_DATASET_PATH = SIMPLE_DIR / "dataset_real_spb_simple.json"
SIMPLE_SUMMARY_PATH = SIMPLE_DIR / "summary_real_spb_simple.json"

if not SIMPLE_DATASET_PATH.exists():
    _summary = build_real_simple_dataset(
        base_dataset_path=BASE_DATASET_PATH,
        out_dataset_path=SIMPLE_DATASET_PATH,
        out_summary_path=SIMPLE_SUMMARY_PATH,
        config=RealSimpleBuildConfig(max_tasks=32, max_agents=64, seed=42),
    )
    print("real_simple built")
else:
    _summary = json.loads(SIMPLE_SUMMARY_PATH.read_text(encoding="utf-8")) if SIMPLE_SUMMARY_PATH.exists() else {}

DATASET_PATH = SIMPLE_DATASET_PATH
print("BASE_DATASET_PATH:", BASE_DATASET_PATH)
print("DATASET_PATH     :", DATASET_PATH)
print("exists           :", DATASET_PATH.exists())

_summary


BASE_DATASET_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/dataset_real_spb_ready.json
DATASET_PATH     : /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/real_simple/dataset_real_spb_simple.json
exists           : True


{'base_report': {'dataset_path': '/Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/dataset_real_spb_ready.json',
  'basic_checks_ok': True,
  'reachability_ok': True,
  'compatibility_ok': True,
  'all_checks_ok': True,
  'counts': {'nodes': 46730,
   'edges': 127100,
   'agents': 626,
   'tasks': 220,
   'routes': 0},
  'issues': {'basic_errors': [],
   'schema_errors': [],
   'uncovered_tasks': [],
   'reachability': {'weakly_connected_components': 1,
    'largest_weak_component_size': 46730,
    'strongly_connected_components': 1,
    'largest_strong_component_size': 46730,
    'unreachable_tasks': [],
    'unreachable_special_nodes': []},
   'metadata_keys': ['agent_depots',
    'agent_fraction',
    'bbox',
    'container_to_vehicle_types',
    'counts',
    'depot_node_ids',
    'fraction_tag',
    'is_anonymized',
    'name',
    'preprocess',
    'profile',
    'profile_description',
    'seed',
    'service_hours_by_container',
    'source_description',
    'vehicl

In [23]:
# GAP
GAP_STEP1_METHOD = "lp"
GAP_ITER = 20

# Real MILP
RMILP_TIME_LIMIT_SEC = 60
RMILP_UNASSIGNED_PENALTY = 1e5

# Genetic
GA_POPULATION_SIZE = 40
GA_GENERATIONS = 50
GA_ELITE_SIZE = 4
GA_SEED = 42

# Live progress in notebook output
SHOW_ALGO_PROGRESS = True
SHOW_SOLVER_DETAILS = False

from time import perf_counter

def make_progress_logger(enabled: bool):
    if not enabled:
        return None
    t0 = perf_counter()
    def _log(message: str) -> None:
        dt = perf_counter() - t0
        print(f"[+{dt:7.1f}s] {message}", flush=True)
    return _log

progress_log = make_progress_logger(SHOW_ALGO_PROGRESS)

def run_with_live_status(label, fn, **kwargs):
    if progress_log is not None:
        progress_log(f"{label}: START")
    ts = perf_counter()
    out = fn(**kwargs)
    if progress_log is not None:
        progress_log(f"{label}: DONE in {perf_counter() - ts:.1f}s")
    return out

results = [
    run_with_live_status(
        "GAP-VRP",
        run_real_gap_vrp,
        dataset_path=DATASET_PATH,
        step1_method=GAP_STEP1_METHOD,
        gap_iter=GAP_ITER,
        max_agents=MAX_AGENTS if "MAX_AGENTS" in globals() else None,
        use_repair=True,
        show_progress=SHOW_SOLVER_DETAILS,
        verbose=False,
        progress_hook=progress_log,
    ),
    run_with_live_status(
        "REAL-MILP",
        run_real_milp,
        dataset_path=DATASET_PATH,
        time_limit_sec=RMILP_TIME_LIMIT_SEC,
        unassigned_penalty=RMILP_UNASSIGNED_PENALTY,
        show_progress=SHOW_SOLVER_DETAILS,
        progress_hook=progress_log,
    ),
    run_with_live_status(
        "REAL-GENETIC",
        run_real_genetic,
        dataset_path=DATASET_PATH,
        population_size=GA_POPULATION_SIZE,
        generations=GA_GENERATIONS,
        elite_size=GA_ELITE_SIZE,
        seed=GA_SEED,
        show_progress=SHOW_SOLVER_DETAILS,
        progress_hook=progress_log,
    ),
]

benchmark_df = pd.DataFrame([r.as_dict() for r in results])
benchmark_df = benchmark_df.sort_values(
    by=["all_checks_ok", "transport_work_ton_km", "total_km", "runtime_sec"],
    ascending=[False, True, True, True],
    na_position="last",
).reset_index(drop=True)



# Save benchmark artifact to local JSON
from datetime import datetime

LOCAL_OUT_DIR = REPO_ROOT / "notebooks" / "local" / "real_data_simple"
LOCAL_OUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RESULT_JSON_PATH = LOCAL_OUT_DIR / f"benchmark_{RUN_TAG}.json"

records = benchmark_df.where(pd.notna(benchmark_df), None).to_dict(orient="records")
artifact = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "notebook": "real_simple_3algo_benchmark.ipynb",
    "dataset_path": str(DATASET_PATH),
    "config": {
        "gap_step1_method": GAP_STEP1_METHOD,
        "gap_iter": GAP_ITER,
        "max_agents": MAX_AGENTS if "MAX_AGENTS" in globals() else None,
        "rmilp_time_limit_sec": RMILP_TIME_LIMIT_SEC,
        "rmilp_unassigned_penalty": RMILP_UNASSIGNED_PENALTY,
        "ga_population_size": GA_POPULATION_SIZE,
        "ga_generations": GA_GENERATIONS,
        "ga_elite_size": GA_ELITE_SIZE,
        "ga_seed": GA_SEED,
        "show_algo_progress": SHOW_ALGO_PROGRESS,
        "show_solver_details": SHOW_SOLVER_DETAILS,
    },
    "results": records,
}
RESULT_JSON_PATH.write_text(json.dumps(artifact, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", RESULT_JSON_PATH)

main_cols = [
    "algorithm",
    "feasible",
    "all_checks_ok",
    "assigned_routes",
    "unassigned_tasks",
    "active_agents",
    "transport_work_ton_km",
    "total_km",
    "deadhead_km",
    "deadhead_share_pct",
    "total_hours",
    "runtime_sec",
    "solver_error",
]
benchmark_df[[c for c in main_cols if c in benchmark_df.columns]]

[+    0.0s] GAP-VRP: START
[+    0.0s] [real_gap_vrp] load dataset: /Users/igoreshka/Desktop/Optimization-of-flows/src/data/real/real_simple/dataset_real_spb_simple.json
[+    0.0s] [real_gap_vrp] dataset loaded (nodes=584, edges=1426, agents=64, tasks=32)
[+    0.0s] [real_gap_vrp] build nx graph
[+    0.0s] [real_gap_vrp] solver start
[+    0.0s] [real_gap_vrp] [GAP-VRP] start: GAP-Lagrangean + VRP(NN+2opt) [step1=lp]
[+    0.0s] [real_gap_vrp] [GAP-VRP] step1: task generation
[+    0.1s] [real_gap_vrp] [GAP-VRP] step1 done in 0.0s, tasks=30
[+    0.1s] [real_gap_vrp] [GAP-VRP] step2: GAP Lagrangean, iter=20
[+    0.1s] [real_gap_vrp] [GAP-LR] prepare matrices: agents=64, tasks=30, max_iter=20
[+    0.1s] [real_gap_vrp] [GAP-LR] precompute costs: 6/64 agents
[+    0.1s] [real_gap_vrp] [GAP-LR] precompute costs: 12/64 agents
[+    0.1s] [real_gap_vrp] [GAP-LR] precompute costs: 18/64 agents
[+    0.1s] [real_gap_vrp] [GAP-LR] precompute costs: 24/64 agents
[+    0.1s] [real_gap_vrp] [

,algorithm,feasible,all_checks_ok,assigned_routes,unassigned_tasks,active_agents,transport_work_ton_km,total_km,deadhead_km,deadhead_share_pct,total_hours,runtime_sec
0,real_genetic,True,True,32,0,17,48.203,576.133,391.013,67.869,32.102,4.404
1,real_gap_vrp,True,True,30,0,10,48.207,466.695,292.137,62.597,27.035,10.493
2,real_milp,False,False,31,1,26,NaN,749.519,570.573,76.125,38.959,0.045


In [25]:
# Детали проверок и параметров по каждому алгоритму
detail_cols = [
    "algorithm",
    "checks",
    "solver_error",
    "step1_method",
    "gap_iter",
    "time_limit_sec",
    "unassigned_penalty",
    "population_size",
    "generations",
    "generations_requested",
    "generation_scale",
    "elite_size",
]
benchmark_df[[c for c in detail_cols if c in benchmark_df.columns]]


,algorithm,checks,step1_method,gap_iter,time_limit_sec,unassigned_penalty,population_size,generations,generations_requested,generation_scale,elite_size
0,real_genetic,"{'hard_constraints_ok': True, 'daily_limits_ok': True, 'reachability_ok': True, 'all_tasks_assigned': True, 'mno_coverage_ok': True, 'all_checks_ok': True}",NaN,NaN,NaN,NaN,40.0,5.0,50.0,0.1,4.0
1,real_gap_vrp,"{'hard_constraints_ok': True, 'daily_limits_ok': True, 'reachability_ok': True, 'all_tasks_assigned': True, 'mno_coverage_ok': True, 'all_checks_ok': True}",lp,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,real_milp,"{'hard_constraints_ok': False, 'daily_limits_ok': True, 'reachability_ok': True, 'all_tasks_assigned': False, 'mno_coverage_ok': False, 'all_checks_ok': False}",NaN,NaN,60.0,100000.0,NaN,NaN,NaN,NaN,NaN


In [26]:
from flowopt.reporting import solution_breakdown_tables
from IPython.display import Markdown, display

MAX_AGENTS_TO_SHOW = 30
MAX_TASK_IDS_IN_CELL = 12
MAX_TASK_ROWS_TO_SHOW = 400

for _, row in benchmark_df.iterrows():
    algo = row.get("algorithm", "unknown")
    tables = solution_breakdown_tables(
        row,
        max_agents=MAX_AGENTS_TO_SHOW,
        max_task_ids=MAX_TASK_IDS_IN_CELL,
    )

    display(Markdown(f"### {algo}: решение по агентам"))
    display(tables["summary"])

    if tables["agents"].empty:
        print("No active agents in solution")
        continue

    display(tables["agents"])
    display(Markdown(f"**{algo}: задача -> агент (первые {MAX_TASK_ROWS_TO_SHOW} строк)**"))
    display(tables["tasks"].head(MAX_TASK_ROWS_TO_SHOW))


### real_genetic: решение по агентам

,algorithm,active_agents,assigned_tasks,total_mass_tons,total_km,total_hours
0,real_genetic,17,32,9.96,576.133,32.102


,agent_id,vehicle_type,is_compact,tasks_count,routes_count,total_mass_tons,total_km,deadhead_km,deadhead_share_pct,total_hours,task_ids
0,AGENT_00125,VT_AD,True,5,5,1.335,65.025,39.283,60.412,4.556,"TASK_00109, TASK_00117, TASK_00119, TASK_00124, TASK_00091"
1,AGENT_00055,VT_AB,False,3,3,0.874,33.104,24.815,74.960,2.059,"TASK_00046, TASK_00010, TASK_00001"
2,AGENT_00053,VT_AB,False,3,3,0.505,47.040,30.013,63.802,2.620,"TASK_00041, TASK_00054, TASK_00039"
3,AGENT_00048,VT_AB,False,2,2,1.178,31.909,22.411,70.236,1.790,"TASK_00058, TASK_00068"
4,AGENT_00051,VT_AB,False,2,2,0.359,32.873,21.660,65.889,1.810,"TASK_00049, TASK_00025"
5,AGENT_00056,VT_AB,False,2,2,0.684,33.609,23.700,70.516,1.860,"TASK_00018, TASK_00009"
6,AGENT_00061,VT_AB,False,2,2,0.166,34.043,22.391,65.772,1.858,"TASK_00019, TASK_00043"
7,AGENT_00057,VT_AB,False,2,2,0.504,35.926,25.848,71.947,1.937,"TASK_00032, TASK_00073"
8,AGENT_00020,VT_A,False,2,2,0.526,37.157,24.559,66.094,1.988,"TASK_00079, TASK_00053"
9,AGENT_00050,VT_AB,False,2,2,0.343,37.761,19.600,51.904,2.013,"TASK_00024, TASK_00015"


**real_genetic: задача -> агент (первые 400 строк)**

,algorithm,agent_id,task_order,task_id
0,real_genetic,AGENT_00125,1,TASK_00109
1,real_genetic,AGENT_00125,2,TASK_00117
2,real_genetic,AGENT_00125,3,TASK_00119
3,real_genetic,AGENT_00125,4,TASK_00124
4,real_genetic,AGENT_00125,5,TASK_00091
5,real_genetic,AGENT_00055,1,TASK_00046
6,real_genetic,AGENT_00055,2,TASK_00010
7,real_genetic,AGENT_00055,3,TASK_00001
8,real_genetic,AGENT_00053,1,TASK_00041
9,real_genetic,AGENT_00053,2,TASK_00054


...

### real_milp: решение по агентам

,algorithm,active_agents,assigned_tasks,total_mass_tons,total_km,total_hours
0,real_milp,26,31,9.859,749.519,38.959


,agent_id,vehicle_type,is_compact,tasks_count,routes_count,total_mass_tons,total_km,deadhead_km,deadhead_share_pct,total_hours,task_ids
0,AGENT_00125,VT_AD,True,4,4,1.234,52.678,33.109,62.852,3.674,"TASK_00109, TASK_00117, TASK_00119, TASK_00124"
1,AGENT_00046,VT_AB,True,2,2,0.505,34.840,26.858,77.088,1.892,"TASK_00054, TASK_00043"
2,AGENT_00060,VT_AB,False,2,2,0.193,39.359,21.321,54.170,2.080,"TASK_00020, TASK_00046"
3,AGENT_00055,VT_AB,False,1,1,0.629,25.637,22.348,87.173,1.308,TASK_00009
4,AGENT_00005,VT_A,True,1,1,0.337,25.763,16.777,65.122,1.293,TASK_00069
5,AGENT_00056,VT_AB,False,1,1,0.331,26.143,19.448,74.390,1.309,TASK_00059
6,AGENT_00158,VT_C,False,1,1,0.920,26.402,20.895,79.142,1.550,TASK_00044
7,AGENT_00037,VT_AB,True,1,1,0.173,26.437,21.103,79.824,1.322,TASK_00032
8,AGENT_00010,VT_A,True,1,1,0.359,26.442,20.625,77.999,1.322,TASK_00012
9,AGENT_00059,VT_AB,False,1,1,0.822,26.442,20.625,77.999,1.342,TASK_00002


**real_milp: задача -> агент (первые 400 строк)**

,algorithm,agent_id,task_order,task_id
0,real_milp,AGENT_00125,1,TASK_00109
1,real_milp,AGENT_00125,2,TASK_00117
2,real_milp,AGENT_00125,3,TASK_00119
3,real_milp,AGENT_00125,4,TASK_00124
4,real_milp,AGENT_00046,1,TASK_00054
5,real_milp,AGENT_00046,2,TASK_00043
6,real_milp,AGENT_00060,1,TASK_00020
7,real_milp,AGENT_00060,2,TASK_00046
8,real_milp,AGENT_00055,1,TASK_00009
9,real_milp,AGENT_00005,1,TASK_00069
